# 🎬 keepsfx → giữ lại HIỆU ỨNG (SFX)

Bỏ giọng + nhạc khỏi video, giữ lại tiếng động/hiệu ứng → MP4 (audio = SFX) để lồng tiếng Việt.
Dùng **BandIt** (SOTA, license CC BY-NC — phi thương mại).

**Bước duy nhất:** chọn `Runtime → GPU T4 → Save`, rồi bấm ▶ chạy ô bên dưới (Accept Drive khi hỏi).

In [ ]:
print('=== BUOC 1: Mount Drive ===', flush=True)
from google.colab import drive
drive.mount('/content/drive')

import os, importlib.util, shutil
print('=== BUOC 2: Clone code ===', flush=True)
os.chdir('/content')
!rm -rf /content/keepsfx /content/bandit
!git clone -q https://github.com/yaylbanh/keepsfx.git
!git clone -q https://github.com/kwatcharasupat/bandit.git

print('=== BUOC 3: Cai thu vien (lan dau lau vai phut) ===', flush=True)
if not all(importlib.util.find_spec(m) for m in ['gradio', 'lightning', 'asteroid', 'fire', 'librosa']):
    !pip install -r /content/keepsfx/requirements.txt
    print('=== Cai xong ===', flush=True)
else:
    print('=== Thu vien da co, bo qua ===', flush=True)

print('=== BUOC 4: Chuan bi model BandIt ===', flush=True)
# Toc do <-> chat luong (nhe = nhanh):
#   dnr-3s-mus64-l1snr / dnr-3s-erb48-l1snr  (NHANH nhat, hoi kem)
#   dnr-3s-bark48-l1snr  (CAN BANG - mac dinh)
#   dnr-3s-bark64-l1snr  (sach nhat nhung RAT CHAM tren T4)
MODEL_STEM = 'dnr-3s-bark48-l1snr'
MODELS = '/content/drive/MyDrive/keepsfx_models'
CKPT_DIR = os.path.join(MODELS, 'ckpt')
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT = os.path.join(CKPT_DIR, MODEL_STEM + '.ckpt')
if not os.path.isfile(CKPT):
    print(f'[*] Tai {MODEL_STEM} ...', flush=True)
    !wget -q -O "{CKPT}" "https://zenodo.org/records/10160698/files/{MODEL_STEM}.ckpt?download=1"
else:
    print('[*] Checkpoint da co, bo qua tai.', flush=True)
shutil.copyfile(f'/content/bandit/expt/{MODEL_STEM}.yaml', os.path.join(MODELS, 'hparams.yaml'))

print('=== BUOC 5: Khoi dong app ===', flush=True)
os.environ['BANDIT_DIR'] = '/content/bandit'
os.environ['BANDIT_CKPT'] = CKPT
os.environ['KEEPSFX_OUTPUT'] = '/content/drive/MyDrive/keepsfx_output'
os.environ['KEEPSFX_SHARE'] = '0'
os.chdir('/content/keepsfx')
%run app.py